In [1]:
TEST_CSV_PATH  = "./testSubs.csv"
OUTPUT_DIR = "./finetuned_deepseekr1_model"
INSTRUCTION = "Given these movie subtitles, write a faithful plot summary in one paragraph. Use only information supported by the subtitles."

In [2]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import pandas as pd

gc.collect()
torch.cuda.empty_cache()
tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
tuned_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tuned_pipe = pipeline("text-generation", model=tuned_model, tokenizer=tuned_tokenizer)

test_df = pd.read_csv(TEST_CSV_PATH).dropna(subset=["subtitles"])
print(f"Test set loaded: {len(test_df)} rows")

def get_subtitle_excerpt(subtitle, max_chars=12000):
    subtitle = str(subtitle)
    if len(subtitle) <= max_chars:
        return subtitle
    half = max_chars // 2
    return subtitle[:half] + "\n...\n" + subtitle[-half:]
import re

def generate_summary(subtitle_text):
    prompt = (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Subtitles:\n{get_subtitle_excerpt(subtitle_text, max_chars=12000)}\n\n"
        f"### Summary:\n"
    )

    result = tuned_pipe(
        prompt,
        do_sample=False,    
        max_new_tokens=500,      
        repetition_penalty=1.1,  
        eos_token_id=tuned_tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"].strip()

    if "</think>" in result:
        result = result.split("</think>")[-1].strip()

    if "###" in result:
        result = result.split("###")[0].strip()

    result = re.sub(r'\*\*[^*]+\*\*:?\s*\n?', '', result).strip()
    result = re.sub(r'\n+', ' ', result).strip()

    return result
sample = test_df.head(5).reset_index(drop=True)
print("-" * 80)
for i, row in sample.iterrows():
    print(f"\n[{i+1}] Movie ID: {row['movie_id']}")
    print(generate_summary(row["subtitles"]))
    print("-" * 80)


`torch_dtype` is deprecated! Use `dtype` instead!
/ix1/cs1671-2026s/class_env/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test set loaded: 397 rows
--------------------------------------------------------------------------------

[1] Movie ID: 9249578


/ihome/infsci2440-2026s/rab527/.local/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


The story centers on Pazu, a young boy, and Sheeta, a mysterious girl who falls from the sky. Together, they navigate challenges as Pazu attempts to build a plane to reach Laputa, a floating island. Along the way, they encounter pirates led by Muska, who seek the crystal Sheeta possesses. Through various conflicts and a magical spell, Pazu and Sheeta manage to save Laputa from Muska's control, ensuring its restoration and peace.
--------------------------------------------------------------------------------

[2] Movie ID: 9226114
The re-opening of Fortune Market after a six-year closure brings together a group of characters, including Fishman, Chickman, and Tofu Ping, who collaborate to restore the market's vibrancy. Fishman, once a vendor, now manages the market, while Chickman, a former customer turned owner, leads the effort. Challenges include organizing vendors, attracting customers, and addressing the market's decline. Personal relationships, especially Fishman's support for Miu

In [16]:
import os
from tqdm import tqdm

test_df = pd.read_csv(TEST_CSV_PATH).dropna(subset=["subtitles"]).reset_index(drop=True)
print(f"Generating summaries for {len(test_df)} movies...")

if os.path.exists("deepseek_finetuned_summaries.csv"):
    completed_df = pd.read_csv("deepseek_finetuned_summaries.csv")
    completed_ids = set(completed_df["movie_id"].tolist())
    results = completed_df.to_dict("records")
    print(f"Resuming — {len(completed_ids)} already done")
else:
    completed_ids = set()
    results = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if row["movie_id"] in completed_ids:
        continue
    results.append({"movie_id": row["movie_id"], "generated_summary": generate_summary(row["subtitles"])})
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv("deepseek_finetuned_summaries.csv", index=False)
        print(f"Checkpoint at {i+1}")

pd.DataFrame(results).to_csv("deepseek_finetuned_summaries.csv", index=False)
print(f"Done. {len(results)} summaries saved.")

Generating summaries for 397 movies...


  0%|          | 0/397 [00:00<?, ?it/s]/ihome/infsci2440-2026s/rab527/.local/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
 13%|█▎        | 50/397 [10:22<1:11:38, 12.39s/it]

Checkpoint at 50


 25%|██▌       | 100/397 [20:23<1:02:18, 12.59s/it]

Checkpoint at 100


...

Checkpoint at 300


 88%|████████▊ | 350/397 [1:12:35<08:40, 11.08s/it]

Checkpoint at 350


100%|██████████| 397/397 [1:22:43<00:00, 12.50s/it]

Done. 397 summaries saved.
